In [5]:
import os
import re
import pandas as pd

folders = {
    "挑": "/home/sabrina/AI-Badminton-111/挑cut",
    "推": "/home/sabrina/AI-Badminton-111/推cut"
}

rows = []

for stroke_type, folder in folders.items():
    for f in os.listdir(folder):
        if not f.endswith(".csv"):
            continue

        # 例：10281挑_1.csv / 10284挑補_12.csv / 10281推_3.csv
        m = re.match(r"^(\d+)(挑|推)(補)?_(\d+)\.csv$", f)

        if m:
            subject_id = m.group(1)
            shot_id = int(m.group(4))

            rows.append({
                "file_name": f,
                "subject_id": subject_id,
                "stroke_type_x": stroke_type,
                "shot_id": shot_id,
                "path": os.path.join(folder, f)
            })

cut_df = pd.DataFrame(rows)

display(cut_df.head())
print("總筆數:", cut_df.shape)
print("挑/推數量:")
print(cut_df["stroke_type_x"].value_counts())

,file_name,subject_id,stroke_type_x,shot_id,path
0,110117挑_7.csv,110117,挑,7,/home/sabrina/AI-Badminton-111/挑cut/110117挑_7.csv
1,110114挑_8.csv,110114,挑,8,/home/sabrina/AI-Badminton-111/挑cut/110114挑_8.csv
2,10305挑_13.csv,10305,挑,13,/home/sabrina/AI-Badminton-111/挑cut/10305挑_13.csv
3,110132挑_5.csv,110132,挑,5,/home/sabrina/AI-Badminton-111/挑cut/110132挑_5.csv
4,110131挑_7.csv,110131,挑,7,/home/sabrina/AI-Badminton-111/挑cut/110131挑_7.csv


總筆數: (3488, 5)
挑/推數量:
stroke_type_x
推    1767
挑    1721
Name: count, dtype: int64


In [6]:
score_df = pd.read_csv("merged_scores_imu.csv")

score_cols = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

# 從 file_name 抓受試者編號，例如 10281挑 -> 10281
score_df["subject_id"] = score_df["file_name"].astype(str).str.extract(r"^(\d+)")

# 只保留 merge 需要的欄位
score_small = score_df[
    ["subject_id", "stroke_type_x"] + score_cols
].drop_duplicates()

merged_df = cut_df.merge(
    score_small,
    on=["subject_id", "stroke_type_x"],
    how="left"
)

display(merged_df.head())
print("merged_df:", merged_df.shape)

print("缺失值檢查:")
print(merged_df[score_cols].isna().sum())

,file_name,subject_id,stroke_type_x,shot_id,path,揮拍軌跡正確度,揮拍速度流暢度,手腕轉動時機正確度,擊球時機正確度,擊球位置正確度
0,110117挑_7.csv,110117,挑,7,/home/sabrina/AI-Badminton-111/挑cut/110117挑_7.csv,1.0,2.0,1.0,3.0,2.0
1,110114挑_8.csv,110114,挑,8,/home/sabrina/AI-Badminton-111/挑cut/110114挑_8.csv,3.0,3.0,3.0,3.0,4.0
2,10305挑_13.csv,10305,挑,13,/home/sabrina/AI-Badminton-111/挑cut/10305挑_13.csv,2.0,5.0,3.0,5.0,1.0
3,110132挑_5.csv,110132,挑,5,/home/sabrina/AI-Badminton-111/挑cut/110132挑_5.csv,1.0,2.0,1.0,4.0,3.0
4,110131挑_7.csv,110131,挑,7,/home/sabrina/AI-Badminton-111/挑cut/110131挑_7.csv,3.0,4.0,2.0,4.0,5.0


merged_df: (3488, 10)
缺失值檢查:
揮拍軌跡正確度      86
揮拍速度流暢度      86
手腕轉動時機正確度    86
擊球時機正確度      86
擊球位置正確度      86
dtype: int64


In [7]:
missing = merged_df[merged_df[score_cols[0]].isna()]

display(missing[["file_name", "subject_id", "stroke_type_x", "shot_id", "path"]].head(100))

print(missing.groupby(["stroke_type_x", "subject_id"]).size())

,file_name,subject_id,stroke_type_x,shot_id,path
39,11016挑_28.csv,11016,挑,28,/home/sabrina/AI-Badminton-111/挑cut/11016挑_28.csv
84,11016挑_27.csv,11016,挑,27,/home/sabrina/AI-Badminton-111/挑cut/11016挑_27.csv
106,11016挑_5.csv,11016,挑,5,/home/sabrina/AI-Badminton-111/挑cut/11016挑_5.csv
230,11016挑_13.csv,11016,挑,13,/home/sabrina/AI-Badminton-111/挑cut/11016挑_13.csv
250,11016挑_2.csv,11016,挑,2,/home/sabrina/AI-Badminton-111/挑cut/11016挑_2.csv
...,...,...,...,...,...
3349,110129推_27.csv,110129,推,27,/home/sabrina/AI-Badminton-111/推cut/110129推_27...
3375,110129推_14.csv,110129,推,14,/home/sabrina/AI-Badminton-111/推cut/110129推_14...
3398,110129推_16.csv,110129,推,16,/home/sabrina/AI-Badminton-111/推cut/110129推_16...
3465,110129推_4.csv,110129,推,4,/home/sabrina/AI-Badminton-111/推cut/110129推_4.csv


stroke_type_x  subject_id
挑              11016         28
推              110129        28
               110130        30
dtype: int64


- 挑 11016：28 筆
- 推 110129：28 筆
- 推 110130：30 筆
- 總共 86 筆 沒有對應分數

In [8]:
merged_df = merged_df.dropna(subset=score_cols).reset_index(drop=True)

print("刪掉缺評分後:", merged_df.shape)
print("缺失值檢查:")
print(merged_df[score_cols].isna().sum())

print("挑/推數量:")
print(merged_df["stroke_type_x"].value_counts())

刪掉缺評分後: (3402, 10)
缺失值檢查:
揮拍軌跡正確度      0
揮拍速度流暢度      0
手腕轉動時機正確度    0
擊球時機正確度      0
擊球位置正確度      0
dtype: int64
挑/推數量:
stroke_type_x
推    1709
挑    1693
Name: count, dtype: int64


In [9]:
sample_path = merged_df.iloc[0]["path"]

sample_df = pd.read_csv(sample_path, header=None)

sample_df.columns = [
    "t1",
    "acc_x", "acc_y", "acc_z",
    "t2",
    "gyro_x", "gyro_y", "gyro_z"
]

display(sample_df.head())
print(sample_df.shape)
print(sample_df.columns)

def load_imu_csv(csv_path):
    df = pd.read_csv(csv_path, header=None)

    df.columns = [
        "t1",
        "acc_x", "acc_y", "acc_z",
        "t2",
        "gyro_x", "gyro_y", "gyro_z"
    ]

    return df[[
        "acc_x", "acc_y", "acc_z",
        "gyro_x", "gyro_y", "gyro_z"
    ]]

,t1,acc_x,acc_y,acc_z,t2,gyro_x,gyro_y,gyro_z
0,25850.0,3.666,7.072,4.974,25850.0,-0.654,0.338,0.392
1,25860.0,3.728,7.110,5.128,25860.0,-0.346,0.346,0.410
2,25870.0,3.622,7.226,5.110,25870.0,-0.100,0.338,0.400
3,25880.0,3.454,7.176,5.290,25880.0,0.026,0.334,0.410
4,25890.0,3.464,7.052,5.242,25890.0,0.088,0.370,0.470


(125, 8)
Index(['t1', 'acc_x', 'acc_y', 'acc_z', 't2', 'gyro_x', 'gyro_y', 'gyro_z'], dtype='object')


In [10]:
import numpy as np

X_list = []

for _, row in merged_df.iterrows():
    imu_df = load_imu_csv(row["path"])
    X_list.append(imu_df.values)

X = np.array(X_list)

print("X shape:", X.shape)

X shape: (3402, 125, 6)


In [11]:
def convert_score_to_category(score):
    score = int(score)
    return (score - 1) // 2   # 1,2 -> 0 ; 3,4 -> 1 ; 5 -> 2

score_cols = [
    "揮拍軌跡正確度",
    "揮拍速度流暢度",
    "手腕轉動時機正確度",
    "擊球時機正確度",
    "擊球位置正確度"
]

for col in score_cols:
    merged_df[col + "_cat"] = merged_df[col].apply(convert_score_to_category)

y1 = merged_df["揮拍軌跡正確度_cat"].values
y2 = merged_df["揮拍速度流暢度_cat"].values
y3 = merged_df["手腕轉動時機正確度_cat"].values
y4 = merged_df["擊球時機正確度_cat"].values
y5 = merged_df["擊球位置正確度_cat"].values

print("y1:", y1.shape, np.unique(y1, return_counts=True))
print("y2:", y2.shape, np.unique(y2, return_counts=True))
print("y3:", y3.shape, np.unique(y3, return_counts=True))
print("y4:", y4.shape, np.unique(y4, return_counts=True))
print("y5:", y5.shape, np.unique(y5, return_counts=True))

y1: (3402,) (array([0, 1, 2]), array([ 468, 1264, 1670]))
y2: (3402,) (array([0, 1, 2]), array([ 588, 1525, 1289]))
y3: (3402,) (array([0, 1, 2]), array([ 711, 1316, 1375]))
y4: (3402,) (array([0, 1, 2]), array([ 531, 1582, 1289]))
y5: (3402,) (array([0, 1, 2]), array([ 317, 1482, 1603]))


In [12]:
挑_idx = merged_df["stroke_type_x"] == "挑"
推_idx = merged_df["stroke_type_x"] == "推"

X_挑 = X[挑_idx]
X_推 = X[推_idx]

print("挑球:", X_挑.shape)
print("平推:", X_推.shape)

print("挑球人數:", merged_df[挑_idx]["subject_id"].nunique())
print("平推人數:", merged_df[推_idx]["subject_id"].nunique())

挑球: (1693, 125, 6)
平推: (1709, 125, 6)
挑球人數: 59
平推人數: 58


## 新的 fine tune，揮拍軌跡正確度

In [15]:
# import numpy as np
# import pandas as pd
# import tensorflow as tf

# from sklearn.model_selection import GroupShuffleSplit
# from sklearn.metrics import accuracy_score

# np.random.seed(42)
# tf.random.set_seed(42)

# # =============================
# # 只先跑 model1：揮拍軌跡正確度
# # =============================
# metric_name = "揮拍軌跡正確度"
# model_path = "專題/models/model1.keras"
# y_all = y1

# ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
# repeat_times = 3
# epochs = 30

# # 從 file_name 拆出受試者 ID
# # 例如 h001_44333_1 -> h001
# merged_df["subject_id"] = merged_df["file_name"].astype(str).str.split("_").str[0]

# # =============================
# # Scratch model
# # =============================
# def build_scratch_model(input_shape=(125, 6), num_classes=3):
#     model = tf.keras.Sequential([
#         tf.keras.layers.Input(shape=input_shape),
#         tf.keras.layers.LSTM(16),
#         tf.keras.layers.Dropout(0.2),
#         tf.keras.layers.Dense(num_classes, activation="softmax")
#     ])

#     model.compile(
#         optimizer="adam",
#         loss="sparse_categorical_crossentropy",
#         metrics=["accuracy"]
#     )

#     return model


# # =============================
# # Transfer model
# # =============================
# def build_transfer_model(base_model, num_classes=3):
#     model = tf.keras.models.clone_model(base_model)
#     model.set_weights(base_model.get_weights())

#     model.pop()  # 拿掉原本最後一層分類頭

#     for layer in model.layers:
#         layer.trainable = False

#     model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(1e-4),
#         loss="sparse_categorical_crossentropy",
#         metrics=["accuracy"]
#     )

#     return model


# # =============================
# # 以人為單位切 train/test
# # =============================
# def subject_group_split(X_data, y_data, groups, test_size=0.2, seed=42):
#     splitter = GroupShuffleSplit(
#         n_splits=1,
#         test_size=test_size,
#         random_state=seed
#     )

#     train_idx, test_idx = next(
#         splitter.split(X_data, y_data, groups=groups)
#     )

#     return train_idx, test_idx


# # =============================
# # 從 train pool 裡面再以人為單位抽小資料
# # =============================
# def sample_subject_ratio(train_idx, groups_all, ratio, seed=42):
#     rng = np.random.default_rng(seed)

#     train_subjects = np.array(sorted(set(groups_all.iloc[train_idx])))

#     n_subjects = max(1, int(len(train_subjects) * ratio))

#     sampled_subjects = rng.choice(
#         train_subjects,
#         size=n_subjects,
#         replace=False
#     )

#     sampled_idx = [
#         idx for idx in train_idx
#         if groups_all.iloc[idx] in sampled_subjects
#     ]

#     return np.array(sampled_idx)


# # =============================
# # Load pretrained model
# # =============================
# base_model_original = tf.keras.models.load_model(model_path)

# all_results = []

# for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

#     print("正在跑:", metric_name, "-", stroke_name)

#     X_target = X[idx]
#     y_target = y_all[idx]

#     target_df = merged_df[idx].reset_index(drop=True)
#     groups = target_df["subject_id"]

#     # 先以人為單位切 train/test
#     train_idx, test_idx = subject_group_split(
#         X_target,
#         y_target,
#         groups,
#         test_size=0.2,
#         seed=42
#     )

#     X_test = X_target[test_idx]
#     y_test = y_target[test_idx]

#     print("train subjects:", groups.iloc[train_idx].nunique())
#     print("test subjects:", groups.iloc[test_idx].nunique())
#     print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

#     for r in ratios:

#         scratch_scores = []
#         transfer_scores = []

#         for run in range(repeat_times):

#             seed = 42 + run

#             small_idx = sample_subject_ratio(
#                 train_idx=train_idx,
#                 groups_all=groups,
#                 ratio=r,
#                 seed=seed
#             )

#             X_small = X_target[small_idx]
#             y_small = y_target[small_idx]

#             print(
#                 f"{stroke_name} | ratio {int(r*100)}% | run {run+1} | "
#                 f"train samples: {len(X_small)} | "
#                 f"train subjects: {groups.iloc[small_idx].nunique()}"
#             )

#             early_stop = tf.keras.callbacks.EarlyStopping(
#                 monitor="loss",
#                 patience=5,
#                 restore_best_weights=True
#             )

#             # ---- Scratch ----
#             tf.keras.backend.clear_session()
#             scratch_model = build_scratch_model(
#                 input_shape=X_target.shape[1:]
#             )

#             scratch_model.fit(
#                 X_small,
#                 y_small,
#                 epochs=epochs,
#                 batch_size=8,
#                 verbose=0,
#                 callbacks=[early_stop]
#             )

#             pred_scratch = scratch_model.predict(X_test, verbose=0)
#             acc_scratch = accuracy_score(
#                 y_test,
#                 np.argmax(pred_scratch, axis=1)
#             )

#             scratch_scores.append(acc_scratch)

#             # ---- Transfer ----
#             tf.keras.backend.clear_session()
#             transfer_model = build_transfer_model(base_model_original)

#             transfer_model.fit(
#                 X_small,
#                 y_small,
#                 epochs=epochs,
#                 batch_size=8,
#                 verbose=0,
#                 callbacks=[early_stop]
#             )

#             pred_transfer = transfer_model.predict(X_test, verbose=0)
#             acc_transfer = accuracy_score(
#                 y_test,
#                 np.argmax(pred_transfer, axis=1)
#             )

#             transfer_scores.append(acc_transfer)

#         all_results.append({
#             "維度": metric_name,
#             "動作": stroke_name,
#             "資料比例": f"{int(r*100)}%",
#             "Scratch_mean": np.mean(scratch_scores),
#             "Scratch_std": np.std(scratch_scores),
#             "Transfer_mean": np.mean(transfer_scores),
#             "Transfer_std": np.std(transfer_scores),
#             "測試方式": "subject-based split"
#         })

# result_df = pd.DataFrame(all_results)
# display(result_df)

# result_df.to_csv(
#     f"{metric_name}_以人為單位(新)_finetune_.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

## 新的 fine tune, 揮拍速度流暢度

In [14]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# 只先跑 model2：揮拍速度流暢度
# =============================
metric_name = "揮拍速度流暢度"
model_path = "專題/models/model2.keras"
y_all = y2

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# 從 file_name 拆出受試者 ID
# 例如 h001_44333_1 -> h001
merged_df["subject_id"] = merged_df["file_name"].astype(str).str.split("_").str[0]

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()  # 拿掉原本最後一層分類頭

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# 以人為單位切 train/test
# =============================
def subject_group_split(X_data, y_data, groups, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed
    )

    train_idx, test_idx = next(
        splitter.split(X_data, y_data, groups=groups)
    )

    return train_idx, test_idx


# =============================
# 從 train pool 裡面再以人為單位抽小資料
# =============================
def sample_subject_ratio(train_idx, groups_all, ratio, seed=42):
    rng = np.random.default_rng(seed)

    train_subjects = np.array(sorted(set(groups_all.iloc[train_idx])))

    n_subjects = max(1, int(len(train_subjects) * ratio))

    sampled_subjects = rng.choice(
        train_subjects,
        size=n_subjects,
        replace=False
    )

    sampled_idx = [
        idx for idx in train_idx
        if groups_all.iloc[idx] in sampled_subjects
    ]

    return np.array(sampled_idx)


# =============================
# Load pretrained model
# =============================
base_model_original = tf.keras.models.load_model(model_path)

all_results = []

for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

    print("正在跑:", metric_name, "-", stroke_name)

    X_target = X[idx]
    y_target = y_all[idx]

    target_df = merged_df[idx].reset_index(drop=True)
    groups = target_df["subject_id"]

    # 先以人為單位切 train/test
    train_idx, test_idx = subject_group_split(
        X_target,
        y_target,
        groups,
        test_size=0.2,
        seed=42
    )

    X_test = X_target[test_idx]
    y_test = y_target[test_idx]

    print("train subjects:", groups.iloc[train_idx].nunique())
    print("test subjects:", groups.iloc[test_idx].nunique())
    print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

    for r in ratios:

        scratch_scores = []
        transfer_scores = []

        for run in range(repeat_times):

            seed = 42 + run

            small_idx = sample_subject_ratio(
                train_idx=train_idx,
                groups_all=groups,
                ratio=r,
                seed=seed
            )

            X_small = X_target[small_idx]
            y_small = y_target[small_idx]

            print(
                f"{stroke_name} | ratio {int(r*100)}% | run {run+1} | "
                f"train samples: {len(X_small)} | "
                f"train subjects: {groups.iloc[small_idx].nunique()}"
            )

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="loss",
                patience=5,
                restore_best_weights=True
            )

            # ---- Scratch ----
            tf.keras.backend.clear_session()
            scratch_model = build_scratch_model(
                input_shape=X_target.shape[1:]
            )

            scratch_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_scratch = scratch_model.predict(X_test, verbose=0)
            acc_scratch = accuracy_score(
                y_test,
                np.argmax(pred_scratch, axis=1)
            )

            scratch_scores.append(acc_scratch)

            # ---- Transfer ----
            tf.keras.backend.clear_session()
            transfer_model = build_transfer_model(base_model_original)

            transfer_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_transfer = transfer_model.predict(X_test, verbose=0)
            acc_transfer = accuracy_score(
                y_test,
                np.argmax(pred_transfer, axis=1)
            )

            transfer_scores.append(acc_transfer)

        all_results.append({
            "維度": metric_name,
            "動作": stroke_name,
            "資料比例": f"{int(r*100)}%",
            "Scratch_mean": np.mean(scratch_scores),
            "Scratch_std": np.std(scratch_scores),
            "Transfer_mean": np.mean(transfer_scores),
            "Transfer_std": np.std(transfer_scores),
            "測試方式": "subject-based split"
        })

result_df = pd.DataFrame(all_results)
display(result_df)

result_df.to_csv(
    f"{metric_name}_以人為單位(新)_finetune_.csv",
    index=False,
    encoding="utf-8-sig"
)

I0000 00:00:1778738734.390041    6775 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778738734.408251    6775 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778738736.121651    6775 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778738741.238242    6775 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

正在跑: 揮拍速度流暢度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | ratio 10% | run 1 | train samples: 117 | train subjects: 4
挑球 | ratio 10% | run 2 | train samples: 106 | train subjects: 4
挑球 | ratio 10% | run 3 | train samples: 122 | train subjects: 4
挑球 | ratio 20% | run 1 | train samples: 272 | train subjects: 9
挑球 | ratio 20% | run 2 | train samples: 240 | train subjects: 9
挑球 | ratio 20% | run 3 | train samples: 260 | train subjects: 9
挑球 | ratio 30% | run 1 | train samples: 411 | train subjects: 14
挑球 | ratio 30% | run 2 | train samples: 393 | train subjects: 14
挑球 | ratio 30% | run 3 | train samples: 412 | train subjects: 14
挑球 | ratio 40% | run 1 | train samples: 527 | train subjects: 18
挑球 | ratio 40% | run 2 | train samples: 535 | train subjects: 18
挑球 | ratio 40% | run 3 | train samples: 532 | train subjects: 18
挑球 | ratio 50% | run 1 | train samples: 655 | train subjects: 23
挑球 | ratio 50% | run 2 | train samples: 688 | train subjects: 23
挑球 | ratio 50% | run 3 | t

,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,測試方式
0,揮拍速度流暢度,挑球,10%,0.304843,0.015254,0.367521,0.030419,subject-based split
1,揮拍速度流暢度,挑球,20%,0.415954,0.112839,0.332384,0.055163,subject-based split
2,揮拍速度流暢度,挑球,30%,0.347578,0.032233,0.345679,0.025518,subject-based split
3,揮拍速度流暢度,挑球,40%,0.341880,0.097395,0.342830,0.056327,subject-based split
4,揮拍速度流暢度,挑球,50%,0.423552,0.108070,0.370370,0.075270,subject-based split
5,揮拍速度流暢度,平推,10%,0.383187,0.138428,0.376344,0.101990,subject-based split
6,揮拍速度流暢度,平推,20%,0.516129,0.089367,0.330401,0.176186,subject-based split
7,揮拍速度流暢度,平推,30%,0.414467,0.063184,0.438905,0.083301,subject-based split
8,揮拍速度流暢度,平推,40%,0.397849,0.063456,0.468231,0.065194,subject-based split
9,揮拍速度流暢度,平推,50%,0.434995,0.011316,0.346041,0.131737,subject-based split


## 新的 fine tune，手腕轉動時機正確度

In [16]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# 只先跑 model3：手腕轉動時機正確度
# =============================
metric_name = "手腕轉動時機正確度"
model_path = "專題/models/model3.keras"
y_all = y3

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# 從 file_name 拆出受試者 ID
# 例如 h001_44333_1 -> h001
merged_df["subject_id"] = merged_df["file_name"].astype(str).str.split("_").str[0]

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()  # 拿掉原本最後一層分類頭

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# 以人為單位切 train/test
# =============================
def subject_group_split(X_data, y_data, groups, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed
    )

    train_idx, test_idx = next(
        splitter.split(X_data, y_data, groups=groups)
    )

    return train_idx, test_idx


# =============================
# 從 train pool 裡面再以人為單位抽小資料
# =============================
def sample_subject_ratio(train_idx, groups_all, ratio, seed=42):
    rng = np.random.default_rng(seed)

    train_subjects = np.array(sorted(set(groups_all.iloc[train_idx])))

    n_subjects = max(1, int(len(train_subjects) * ratio))

    sampled_subjects = rng.choice(
        train_subjects,
        size=n_subjects,
        replace=False
    )

    sampled_idx = [
        idx for idx in train_idx
        if groups_all.iloc[idx] in sampled_subjects
    ]

    return np.array(sampled_idx)


# =============================
# Load pretrained model
# =============================
base_model_original = tf.keras.models.load_model(model_path)

all_results = []

for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

    print("正在跑:", metric_name, "-", stroke_name)

    X_target = X[idx]
    y_target = y_all[idx]

    target_df = merged_df[idx].reset_index(drop=True)
    groups = target_df["subject_id"]

    # 先以人為單位切 train/test
    train_idx, test_idx = subject_group_split(
        X_target,
        y_target,
        groups,
        test_size=0.2,
        seed=42
    )

    X_test = X_target[test_idx]
    y_test = y_target[test_idx]

    print("train subjects:", groups.iloc[train_idx].nunique())
    print("test subjects:", groups.iloc[test_idx].nunique())
    print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

    for r in ratios:

        scratch_scores = []
        transfer_scores = []

        for run in range(repeat_times):

            seed = 42 + run

            small_idx = sample_subject_ratio(
                train_idx=train_idx,
                groups_all=groups,
                ratio=r,
                seed=seed
            )

            X_small = X_target[small_idx]
            y_small = y_target[small_idx]

            print(
                f"{stroke_name} | ratio {int(r*100)}% | run {run+1} | "
                f"train samples: {len(X_small)} | "
                f"train subjects: {groups.iloc[small_idx].nunique()}"
            )

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="loss",
                patience=5,
                restore_best_weights=True
            )

            # ---- Scratch ----
            tf.keras.backend.clear_session()
            scratch_model = build_scratch_model(
                input_shape=X_target.shape[1:]
            )

            scratch_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_scratch = scratch_model.predict(X_test, verbose=0)
            acc_scratch = accuracy_score(
                y_test,
                np.argmax(pred_scratch, axis=1)
            )

            scratch_scores.append(acc_scratch)

            # ---- Transfer ----
            tf.keras.backend.clear_session()
            transfer_model = build_transfer_model(base_model_original)

            transfer_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_transfer = transfer_model.predict(X_test, verbose=0)
            acc_transfer = accuracy_score(
                y_test,
                np.argmax(pred_transfer, axis=1)
            )

            transfer_scores.append(acc_transfer)

        all_results.append({
            "維度": metric_name,
            "動作": stroke_name,
            "資料比例": f"{int(r*100)}%",
            "Scratch_mean": np.mean(scratch_scores),
            "Scratch_std": np.std(scratch_scores),
            "Transfer_mean": np.mean(transfer_scores),
            "Transfer_std": np.std(transfer_scores),
            "測試方式": "subject-based split"
        })

result_df = pd.DataFrame(all_results)
display(result_df)

result_df.to_csv(
    f"{metric_name}_以人為單位(新)_finetune_.csv",
    index=False,
    encoding="utf-8-sig"
)

正在跑: 手腕轉動時機正確度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | ratio 10% | run 1 | train samples: 117 | train subjects: 4
挑球 | ratio 10% | run 2 | train samples: 106 | train subjects: 4
挑球 | ratio 10% | run 3 | train samples: 122 | train subjects: 4
挑球 | ratio 20% | run 1 | train samples: 272 | train subjects: 9
挑球 | ratio 20% | run 2 | train samples: 240 | train subjects: 9
挑球 | ratio 20% | run 3 | train samples: 260 | train subjects: 9
挑球 | ratio 30% | run 1 | train samples: 411 | train subjects: 14
挑球 | ratio 30% | run 2 | train samples: 393 | train subjects: 14
挑球 | ratio 30% | run 3 | train samples: 412 | train subjects: 14
挑球 | ratio 40% | run 1 | train samples: 527 | train subjects: 18
挑球 | ratio 40% | run 2 | train samples: 535 | train subjects: 18
挑球 | ratio 40% | run 3 | train samples: 532 | train subjects: 18
挑球 | ratio 50% | run 1 | train samples: 655 | train subjects: 23
挑球 | ratio 50% | run 2 | train samples: 688 | train subjects: 23
挑球 | ratio 50% | run 3 |

,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,測試方式
0,手腕轉動時機正確度,挑球,10%,0.258310,0.088711,0.300095,0.021107,subject-based split
1,手腕轉動時機正確度,挑球,20%,0.428300,0.045127,0.402659,0.013227,subject-based split
2,手腕轉動時機正確度,挑球,30%,0.400760,0.007107,0.321937,0.062980,subject-based split
3,手腕轉動時機正確度,挑球,40%,0.346629,0.061648,0.377018,0.128370,subject-based split
4,手腕轉動時機正確度,挑球,50%,0.263058,0.077931,0.377018,0.067913,subject-based split
5,手腕轉動時機正確度,平推,10%,0.385142,0.114965,0.358749,0.115388,subject-based split
6,手腕轉動時機正確度,平推,20%,0.522972,0.034891,0.394917,0.075427,subject-based split
7,手腕轉動時機正確度,平推,30%,0.495601,0.019599,0.346041,0.102960,subject-based split
8,手腕轉動時機正確度,平推,40%,0.473118,0.034891,0.311828,0.099659,subject-based split
9,手腕轉動時機正確度,平推,50%,0.355816,0.037580,0.431085,0.176018,subject-based split


## 新的 fine tune，擊球時機正確度 

In [19]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# 只先跑 model4：擊球時機正確度
# =============================
metric_name = "擊球時機正確度"
model_path = "專題/models/model4.keras"
y_all = y4

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# 從 file_name 拆出受試者 ID
# 例如 h001_44333_1 -> h001
merged_df["subject_id"] = merged_df["file_name"].astype(str).str.split("_").str[0]

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()  # 拿掉原本最後一層分類頭

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# 以人為單位切 train/test
# =============================
def subject_group_split(X_data, y_data, groups, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed
    )

    train_idx, test_idx = next(
        splitter.split(X_data, y_data, groups=groups)
    )

    return train_idx, test_idx


# =============================
# 從 train pool 裡面再以人為單位抽小資料
# =============================
def sample_subject_ratio(train_idx, groups_all, ratio, seed=42):
    rng = np.random.default_rng(seed)

    train_subjects = np.array(sorted(set(groups_all.iloc[train_idx])))

    n_subjects = max(1, int(len(train_subjects) * ratio))

    sampled_subjects = rng.choice(
        train_subjects,
        size=n_subjects,
        replace=False
    )

    sampled_idx = [
        idx for idx in train_idx
        if groups_all.iloc[idx] in sampled_subjects
    ]

    return np.array(sampled_idx)


# =============================
# Load pretrained model
# =============================
base_model_original = tf.keras.models.load_model(model_path)

all_results = []

for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

    print("正在跑:", metric_name, "-", stroke_name)

    X_target = X[idx]
    y_target = y_all[idx]

    target_df = merged_df[idx].reset_index(drop=True)
    groups = target_df["subject_id"]

    # 先以人為單位切 train/test
    train_idx, test_idx = subject_group_split(
        X_target,
        y_target,
        groups,
        test_size=0.2,
        seed=42
    )

    X_test = X_target[test_idx]
    y_test = y_target[test_idx]

    print("train subjects:", groups.iloc[train_idx].nunique())
    print("test subjects:", groups.iloc[test_idx].nunique())
    print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

    for r in ratios:

        scratch_scores = []
        transfer_scores = []

        for run in range(repeat_times):

            seed = 42 + run

            small_idx = sample_subject_ratio(
                train_idx=train_idx,
                groups_all=groups,
                ratio=r,
                seed=seed
            )

            X_small = X_target[small_idx]
            y_small = y_target[small_idx]

            print(
                f"{stroke_name} | ratio {int(r*100)}% | run {run+1} | "
                f"train samples: {len(X_small)} | "
                f"train subjects: {groups.iloc[small_idx].nunique()}"
            )

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="loss",
                patience=5,
                restore_best_weights=True
            )

            # ---- Scratch ----
            tf.keras.backend.clear_session()
            scratch_model = build_scratch_model(
                input_shape=X_target.shape[1:]
            )

            scratch_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_scratch = scratch_model.predict(X_test, verbose=0)
            acc_scratch = accuracy_score(
                y_test,
                np.argmax(pred_scratch, axis=1)
            )

            scratch_scores.append(acc_scratch)

            # ---- Transfer ----
            tf.keras.backend.clear_session()
            transfer_model = build_transfer_model(base_model_original)

            transfer_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_transfer = transfer_model.predict(X_test, verbose=0)
            acc_transfer = accuracy_score(
                y_test,
                np.argmax(pred_transfer, axis=1)
            )

            transfer_scores.append(acc_transfer)

        all_results.append({
            "維度": metric_name,
            "動作": stroke_name,
            "資料比例": f"{int(r*100)}%",
            "Scratch_mean": np.mean(scratch_scores),
            "Scratch_std": np.std(scratch_scores),
            "Transfer_mean": np.mean(transfer_scores),
            "Transfer_std": np.std(transfer_scores),
            "測試方式": "subject-based split"
        })

result_df = pd.DataFrame(all_results)
display(result_df)

result_df.to_csv(
    f"{metric_name}_以人為單位(新)_finetune_.csv",
    index=False,
    encoding="utf-8-sig"
)

正在跑: 擊球時機正確度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | ratio 10% | run 1 | train samples: 117 | train subjects: 4
挑球 | ratio 10% | run 2 | train samples: 106 | train subjects: 4
挑球 | ratio 10% | run 3 | train samples: 122 | train subjects: 4
挑球 | ratio 20% | run 1 | train samples: 272 | train subjects: 9
挑球 | ratio 20% | run 2 | train samples: 240 | train subjects: 9
挑球 | ratio 20% | run 3 | train samples: 260 | train subjects: 9
挑球 | ratio 30% | run 1 | train samples: 411 | train subjects: 14
挑球 | ratio 30% | run 2 | train samples: 393 | train subjects: 14
挑球 | ratio 30% | run 3 | train samples: 412 | train subjects: 14
挑球 | ratio 40% | run 1 | train samples: 527 | train subjects: 18
挑球 | ratio 40% | run 2 | train samples: 535 | train subjects: 18
挑球 | ratio 40% | run 3 | train samples: 532 | train subjects: 18
挑球 | ratio 50% | run 1 | train samples: 655 | train subjects: 23
挑球 | ratio 50% | run 2 | train samples: 688 | train subjects: 23
挑球 | ratio 50% | run 3 | t

,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,測試方式
0,擊球時機正確度,挑球,10%,0.381766,0.128511,0.268756,0.091917,subject-based split
1,擊球時機正確度,挑球,20%,0.489079,0.100602,0.360874,0.033897,subject-based split
2,擊球時機正確度,挑球,30%,0.474834,0.026146,0.231719,0.142665,subject-based split
3,擊球時機正確度,挑球,40%,0.454891,0.036062,0.314340,0.112631,subject-based split
4,擊球時機正確度,挑球,50%,0.494777,0.006715,0.287749,0.057594,subject-based split
5,擊球時機正確度,平推,10%,0.291300,0.084633,0.457478,0.026663,subject-based split
6,擊球時機正確度,平推,20%,0.473118,0.087399,0.260020,0.063002,subject-based split
7,擊球時機正確度,平推,30%,0.495601,0.079125,0.406647,0.071846,subject-based split
8,擊球時機正確度,平推,40%,0.494624,0.039659,0.274682,0.026699,subject-based split
9,擊球時機正確度,平推,50%,0.356794,0.094764,0.305963,0.040799,subject-based split


## 新的 fine tune，擊球位置正確度

In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# =============================
# 只先跑 model5：擊球位置正確度
# =============================
metric_name = "擊球位置正確度"
model_path = "專題/models/model5.keras"
y_all = y5

ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
repeat_times = 3
epochs = 30

# 從 file_name 拆出受試者 ID
# 例如 h001_44333_1 -> h001
merged_df["subject_id"] = merged_df["file_name"].astype(str).str.split("_").str[0]

# =============================
# Scratch model
# =============================
def build_scratch_model(input_shape=(125, 6), num_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(16),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# Transfer model
# =============================
def build_transfer_model(base_model, num_classes=3):
    model = tf.keras.models.clone_model(base_model)
    model.set_weights(base_model.get_weights())

    model.pop()  # 拿掉原本最後一層分類頭

    for layer in model.layers:
        layer.trainable = False

    model.add(tf.keras.layers.Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =============================
# 以人為單位切 train/test
# =============================
def subject_group_split(X_data, y_data, groups, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=seed
    )

    train_idx, test_idx = next(
        splitter.split(X_data, y_data, groups=groups)
    )

    return train_idx, test_idx


# =============================
# 從 train pool 裡面再以人為單位抽小資料
# =============================
def sample_subject_ratio(train_idx, groups_all, ratio, seed=42):
    rng = np.random.default_rng(seed)

    train_subjects = np.array(sorted(set(groups_all.iloc[train_idx])))

    n_subjects = max(1, int(len(train_subjects) * ratio))

    sampled_subjects = rng.choice(
        train_subjects,
        size=n_subjects,
        replace=False
    )

    sampled_idx = [
        idx for idx in train_idx
        if groups_all.iloc[idx] in sampled_subjects
    ]

    return np.array(sampled_idx)


# =============================
# Load pretrained model
# =============================
base_model_original = tf.keras.models.load_model(model_path)

all_results = []

for stroke_name, idx in [("挑球", 挑_idx), ("平推", 推_idx)]:

    print("正在跑:", metric_name, "-", stroke_name)

    X_target = X[idx]
    y_target = y_all[idx]

    target_df = merged_df[idx].reset_index(drop=True)
    groups = target_df["subject_id"]

    # 先以人為單位切 train/test
    train_idx, test_idx = subject_group_split(
        X_target,
        y_target,
        groups,
        test_size=0.2,
        seed=42
    )

    X_test = X_target[test_idx]
    y_test = y_target[test_idx]

    print("train subjects:", groups.iloc[train_idx].nunique())
    print("test subjects:", groups.iloc[test_idx].nunique())
    print("overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

    for r in ratios:

        scratch_scores = []
        transfer_scores = []

        for run in range(repeat_times):

            seed = 42 + run

            small_idx = sample_subject_ratio(
                train_idx=train_idx,
                groups_all=groups,
                ratio=r,
                seed=seed
            )

            X_small = X_target[small_idx]
            y_small = y_target[small_idx]

            print(
                f"{stroke_name} | ratio {int(r*100)}% | run {run+1} | "
                f"train samples: {len(X_small)} | "
                f"train subjects: {groups.iloc[small_idx].nunique()}"
            )

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="loss",
                patience=5,
                restore_best_weights=True
            )

            # ---- Scratch ----
            tf.keras.backend.clear_session()
            scratch_model = build_scratch_model(
                input_shape=X_target.shape[1:]
            )

            scratch_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_scratch = scratch_model.predict(X_test, verbose=0)
            acc_scratch = accuracy_score(
                y_test,
                np.argmax(pred_scratch, axis=1)
            )

            scratch_scores.append(acc_scratch)

            # ---- Transfer ----
            tf.keras.backend.clear_session()
            transfer_model = build_transfer_model(base_model_original)

            transfer_model.fit(
                X_small,
                y_small,
                epochs=epochs,
                batch_size=8,
                verbose=0,
                callbacks=[early_stop]
            )

            pred_transfer = transfer_model.predict(X_test, verbose=0)
            acc_transfer = accuracy_score(
                y_test,
                np.argmax(pred_transfer, axis=1)
            )

            transfer_scores.append(acc_transfer)

        all_results.append({
            "維度": metric_name,
            "動作": stroke_name,
            "資料比例": f"{int(r*100)}%",
            "Scratch_mean": np.mean(scratch_scores),
            "Scratch_std": np.std(scratch_scores),
            "Transfer_mean": np.mean(transfer_scores),
            "Transfer_std": np.std(transfer_scores),
            "測試方式": "subject-based split"
        })

result_df = pd.DataFrame(all_results)
display(result_df)

result_df.to_csv(
    f"{metric_name}_以人為單位(新)_finetune_.csv",
    index=False,
    encoding="utf-8-sig"
)

正在跑: 擊球位置正確度 - 挑球
train subjects: 47
test subjects: 12
overlap: set()
挑球 | ratio 10% | run 1 | train samples: 117 | train subjects: 4
挑球 | ratio 10% | run 2 | train samples: 106 | train subjects: 4
挑球 | ratio 10% | run 3 | train samples: 122 | train subjects: 4
挑球 | ratio 20% | run 1 | train samples: 272 | train subjects: 9
挑球 | ratio 20% | run 2 | train samples: 240 | train subjects: 9
挑球 | ratio 20% | run 3 | train samples: 260 | train subjects: 9
挑球 | ratio 30% | run 1 | train samples: 411 | train subjects: 14
挑球 | ratio 30% | run 2 | train samples: 393 | train subjects: 14
挑球 | ratio 30% | run 3 | train samples: 412 | train subjects: 14
挑球 | ratio 40% | run 1 | train samples: 527 | train subjects: 18
挑球 | ratio 40% | run 2 | train samples: 535 | train subjects: 18
挑球 | ratio 40% | run 3 | train samples: 532 | train subjects: 18
挑球 | ratio 50% | run 1 | train samples: 655 | train subjects: 23
挑球 | ratio 50% | run 2 | train samples: 688 | train subjects: 23
挑球 | ratio 50% | run 3 | t

,維度,動作,資料比例,Scratch_mean,Scratch_std,Transfer_mean,Transfer_std,測試方式
0,擊球位置正確度,挑球,10%,0.377968,0.033737,0.322887,0.039637,subject-based split
1,擊球位置正確度,挑球,20%,0.408357,0.046601,0.302944,0.049692,subject-based split
2,擊球位置正確度,挑球,30%,0.454891,0.067793,0.337132,0.019784,subject-based split
3,擊球位置正確度,挑球,40%,0.447293,0.081782,0.304843,0.031980,subject-based split
4,擊球位置正確度,挑球,50%,0.464387,0.089732,0.290598,0.061677,subject-based split
5,擊球位置正確度,平推,10%,0.438905,0.169317,0.384164,0.065091,subject-based split
6,擊球位置正確度,平推,20%,0.595308,0.083873,0.279570,0.084667,subject-based split
7,擊球位置正確度,平推,30%,0.486804,0.054127,0.425220,0.120865,subject-based split
8,擊球位置正確度,平推,40%,0.564027,0.069411,0.375367,0.150848,subject-based split
9,擊球位置正確度,平推,50%,0.423265,0.069822,0.335288,0.030788,subject-based split
